# 01 — Preparação de Dataset para Detecção de Violência

Este notebook cobre todo o pipeline de dados:

1. Instalação de dependências
2. Estrutura de diretórios
3. Download do dataset público **RWF-2000** (fight / non-fight)
4. Extração de frames dos vídeos
5. Estimativa de postura agressiva com **MediaPipe Pose**
6. Geração de anotações no formato **YOLO** (3 classes: `person`, `violence`, `pre_violence`)
7. Split treino / validação / teste
8. Geração do `dataset.yaml`
9. Visualização de amostras

---
> **Classes:**
> - `0 — person` : pessoa em comportamento normal
> - `1 — violence` : agressão física ativa (briga, soco, chute)
> - `2 — pre_violence` : postura agressiva / gesto ameaçador (pré-violência)

## 1. Instalação de dependências

In [ ]:
!pip install -q ultralytics opencv-python-headless mediapipe scikit-learn tqdm PyYAML matplotlib

## 2. Imports e configurações globais

In [ ]:
import os
import cv2
import json
import shutil
import zipfile
import urllib.request
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from IPython.display import display, Image as IPImage

# ── Diretórios base ────────────────────────────────────────────────────────────
BASE_DIR      = Path("../data")
RAW_DIR       = BASE_DIR / "raw"          # vídeos brutos
PROCESSED_DIR = BASE_DIR / "processed"    # dataset YOLO
META_DIR      = BASE_DIR / "metadata"

for d in [RAW_DIR / "violence", RAW_DIR / "non_violence",
          PROCESSED_DIR / "images" / "train",
          PROCESSED_DIR / "images" / "val",
          PROCESSED_DIR / "images" / "test",
          PROCESSED_DIR / "labels" / "train",
          PROCESSED_DIR / "labels" / "val",
          PROCESSED_DIR / "labels" / "test",
          META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Configurações ──────────────────────────────────────────────────────────────
CFG = {
    "fps_extract":             5,      # frames por segundo a extrair
    "img_size":                640,    # tamanho de saída das imagens
    "pre_violence_threshold":  0.45,   # score mínimo para rotular como pre_violence
    "train_ratio":             0.70,
    "val_ratio":               0.20,
    "test_ratio":              0.10,
}

# ── Labels ─────────────────────────────────────────────────────────────────────
CLASS_NAMES = {0: "person", 1: "violence", 2: "pre_violence"}
CLASS_COLORS = {0: "green", 1: "red", 2: "orange"}

# MediaPipe Pose
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

print("✓ Ambiente configurado")
print(f"  Dataset será salvo em: {PROCESSED_DIR.resolve()}")

## 3. Download do dataset RWF-2000

O **RWF-2000** (Real-World Fighting) contém 2.000 clipes de vídeo rotulados como `fight` ou `non-fight`.

Opções:
- **A)** Usar seus próprios vídeos (coloque em `data/raw/violence/` e `data/raw/non_violence/`)
- **B)** Baixar via Kaggle API (requer credenciais)
- **C)** Script de download direto (Google Drive / link próprio)

In [ ]:
# ── Opção A: verificar vídeos próprios já colocados nos diretórios ─────────────
fight_videos     = list((RAW_DIR / "violence").glob("*.mp4")) + \
                   list((RAW_DIR / "violence").glob("*.avi")) + \
                   list((RAW_DIR / "violence").glob("*.mov"))
non_fight_videos = list((RAW_DIR / "non_violence").glob("*.mp4")) + \
                   list((RAW_DIR / "non_violence").glob("*.avi")) + \
                   list((RAW_DIR / "non_violence").glob("*.mov"))

print(f"Vídeos de violência encontrados  : {len(fight_videos)}")
print(f"Vídeos sem violência encontrados: {len(non_fight_videos)}")

if len(fight_videos) == 0:
    print("\n⚠ Nenhum vídeo encontrado.")
    print("  Coloque seus vídeos em:")
    print(f"    {(RAW_DIR / 'violence').resolve()}")
    print(f"    {(RAW_DIR / 'non_violence').resolve()}")
    print("  Ou use a célula abaixo para download via Kaggle.")

In [ ]:
# ── Opção B: Download via Kaggle API ──────────────────────────────────────────
# Pré-requisito: arquivo ~/.kaggle/kaggle.json com suas credenciais
# Dataset: https://www.kaggle.com/datasets/vulamnguyen/rwf2000

USE_KAGGLE = False   # ← Altere para True se quiser usar

if USE_KAGGLE:
    !pip install -q kaggle
    !kaggle datasets download -d vulamnguyen/rwf2000 -p {RAW_DIR}
    with zipfile.ZipFile(RAW_DIR / "rwf2000.zip") as z:
        z.extractall(RAW_DIR)
    print("Download concluído.")

## 4. Extração de frames

In [ ]:
def extract_frames(video_path: Path, output_dir: Path, fps: int = 5) -> list:
    """
    Extrai frames de um vídeo à taxa `fps` frames/segundo.
    Retorna lista de caminhos dos frames salvos.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  ⚠ Não foi possível abrir: {video_path}")
        return []

    video_fps      = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_interval = max(1, int(video_fps / fps))
    output_dir.mkdir(parents=True, exist_ok=True)

    saved, frame_idx, saved_idx = [], 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            p = output_dir / f"frame_{saved_idx:05d}.jpg"
            cv2.imwrite(str(p), frame)
            saved.append(p)
            saved_idx += 1
        frame_idx += 1

    cap.release()
    return saved


# Teste rápido com o primeiro vídeo disponível
all_videos = [(v, "violence") for v in fight_videos] + \
             [(v, "non_violence") for v in non_fight_videos]

if all_videos:
    test_vid, test_label = all_videos[0]
    test_frames = extract_frames(test_vid, BASE_DIR / "tmp_test", fps=2)
    print(f"Vídeo: {test_vid.name}  →  {len(test_frames)} frames extraídos")
    shutil.rmtree(BASE_DIR / "tmp_test", ignore_errors=True)
else:
    print("Nenhum vídeo disponível para teste. Adicione vídeos e reexecute.")

## 5. Estimativa de postura agressiva com MediaPipe

Score calculado com 3 sinais de pose:

| Sinal | Descrição |
|---|---|
| Braços erguidos | Pulsos acima dos ombros → ataque |
| Inclinação do tronco | Corpo inclinado para frente → ameaça |
| Abertura dos pulsos | Braços abertos → gesto de soco |

In [ ]:
def compute_pose_features(frame: np.ndarray) -> dict | None:
    """Extrai landmarks da pose com MediaPipe. Retorna None se nenhuma pose for detectada."""
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)
        if not results.pose_landmarks:
            return None
        return {
            "landmarks": [
                {"x": lm.x, "y": lm.y, "z": lm.z, "v": lm.visibility}
                for lm in results.pose_landmarks.landmark
            ]
        }


def estimate_aggression_score(pose: dict) -> float:
    """
    Score de agressividade [0.0 – 1.0] baseado em geometria de pose.
    Quanto maior, mais provável que a cena evoluirá para violência.
    """
    lm = pose["landmarks"]

    # Índices MediaPipe
    L_SH, R_SH   = 11, 12   # ombros
    L_WR, R_WR   = 15, 16   # pulsos
    L_HI, R_HI   = 23, 24   # quadris
    NOSE          = 0

    def pt(i):
        return np.array([lm[i]["x"], lm[i]["y"]])

    try:
        l_sh, r_sh = pt(L_SH), pt(R_SH)
        l_wr, r_wr = pt(L_WR), pt(R_WR)
        l_hi, r_hi = pt(L_HI), pt(R_HI)
        nose       = pt(NOSE)

        # Sinal 1 — braços levantados
        sh_y         = (l_sh[1] + r_sh[1]) / 2
        arm_score    = min(1.0, (max(0, sh_y - l_wr[1]) + max(0, sh_y - r_wr[1])) * 2)

        # Sinal 2 — inclinação do tronco
        torso_vec  = nose - (l_hi + r_hi) / 2
        lean_score = min(1.0, abs(np.arctan2(torso_vec[0], -torso_vec[1])) / (np.pi / 3))

        # Sinal 3 — abertura de pulsos
        spread_score = min(1.0, np.linalg.norm(l_wr - r_wr) * 2)

        return float(np.clip(0.4 * arm_score + 0.3 * lean_score + 0.3 * spread_score, 0, 1))
    except Exception:
        return 0.0


print("✓ Funções de pose definidas")

In [ ]:
# ── Demo visual do score de agressividade ──────────────────────────────────────
# Mostra um frame com a pose desenhada e o score calculado

if all_videos:
    demo_frames = extract_frames(all_videos[0][0], BASE_DIR / "tmp_demo", fps=1)

    fig, axes = plt.subplots(1, min(4, len(demo_frames)), figsize=(16, 4))
    if len(demo_frames) == 1:
        axes = [axes]

    for ax, fp in zip(axes, demo_frames[:4]):
        frame = cv2.imread(str(fp))
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pose  = compute_pose_features(frame)
        score = estimate_aggression_score(pose) if pose else 0.0
        color = "red" if score >= 0.45 else "orange" if score >= 0.25 else "green"

        # Desenha pose no frame
        with mp_pose.Pose(static_image_mode=True) as p:
            res = p.process(rgb)
            if res.pose_landmarks:
                mp_drawing.draw_landmarks(rgb, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        ax.imshow(rgb)
        ax.set_title(f"Score: {score:.2f}", color=color, fontweight="bold")
        ax.axis("off")

    plt.suptitle(f"Vídeo: {all_videos[0][0].name}  |  Label: {all_videos[0][1]}",
                 fontsize=12)
    plt.tight_layout()
    plt.show()
    shutil.rmtree(BASE_DIR / "tmp_demo", ignore_errors=True)

## 6. Detecção de pessoas e geração de anotações YOLO

In [ ]:
from ultralytics import YOLO as YOLOModel

# Modelo leve apenas para detectar bbox de pessoas
_person_detector = YOLOModel("yolov8n.pt")

def detect_person_bboxes(frame: np.ndarray) -> list:
    """
    Retorna lista de bboxes (x1,y1,x2,y2) das pessoas no frame.
    Usa YOLOv8n pré-treinado no COCO (classe 0 = person).
    Fallback: bbox do frame inteiro se ninguém detectado.
    """
    results = _person_detector(frame, classes=[0], verbose=False)[0]
    boxes = [(int(b[0]), int(b[1]), int(b[2]), int(b[3])) for b in results.boxes.xyxy.tolist()]
    return boxes if boxes else [(0, 0, frame.shape[1], frame.shape[0])]


def to_yolo_annotation(w: int, h: int, bbox: tuple, class_id: int) -> str:
    """Converte bbox pixel → formato YOLO normalizado."""
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2 / w
    cy = (y1 + y2) / 2 / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


print("✓ Detector de pessoas carregado (yolov8n.pt)")

## 7. Pipeline completo — processar todos os vídeos

In [ ]:
def process_video(video_path: Path, label: str, cfg: dict) -> list:
    """
    Processa um vídeo e retorna lista de registros por frame.
    Cada registro: frame BGR, class_id, annotations YOLO, aggression_score.
    """
    tmp_dir = BASE_DIR / "tmp" / video_path.stem
    frames  = extract_frames(video_path, tmp_dir, fps=cfg["fps_extract"])
    records = []

    for fp in frames:
        frame = cv2.imread(str(fp))
        if frame is None:
            continue
        h, w = frame.shape[:2]

        # Pose → score de agressividade
        pose  = compute_pose_features(frame)
        score = estimate_aggression_score(pose) if pose else 0.0

        # Definir classe do frame
        if label == "violence":
            class_id = 1          # violence (label do vídeo)
        elif score >= cfg["pre_violence_threshold"]:
            class_id = 2          # pre_violence (detectado por pose)
        else:
            class_id = 0          # person (normal)

        # Bboxes de pessoas
        bboxes      = detect_person_bboxes(frame)
        annotations = [to_yolo_annotation(w, h, b, class_id) for b in bboxes]

        records.append({
            "frame":            frame,
            "video":            video_path.name,
            "label":            label,
            "class_id":         class_id,
            "aggression_score": score,
            "annotations":      annotations,
        })

    shutil.rmtree(tmp_dir, ignore_errors=True)
    return records


# ── Processar todos os vídeos ──────────────────────────────────────────────────
all_records = []

print(f"Processando {len(all_videos)} vídeos...\n")

for video_path, label in tqdm(all_videos, desc="Vídeos"):
    records = process_video(video_path, label, CFG)
    all_records.extend(records)

print(f"\nTotal de frames anotados: {len(all_records)}")
print(f"  violence    : {sum(1 for r in all_records if r['class_id'] == 1)}")
print(f"  pre_violence: {sum(1 for r in all_records if r['class_id'] == 2)}")
print(f"  person      : {sum(1 for r in all_records if r['class_id'] == 0)}")

## 8. Split e escrita do dataset

In [ ]:
# ── Split estratificado ────────────────────────────────────────────────────────
labels_for_split = [r["class_id"] for r in all_records]

train_idx, tmp_idx = train_test_split(
    range(len(all_records)),
    test_size=CFG["val_ratio"] + CFG["test_ratio"],
    stratify=labels_for_split,
    random_state=42,
)
tmp_labels = [labels_for_split[i] for i in tmp_idx]
val_idx, test_idx = train_test_split(
    tmp_idx,
    test_size=CFG["test_ratio"] / (CFG["val_ratio"] + CFG["test_ratio"]),
    stratify=tmp_labels,
    random_state=42,
)

splits = {"train": train_idx, "val": val_idx, "test": test_idx}
print(f"Split:  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")


# ── Escrever imagens e labels ──────────────────────────────────────────────────
IMG_SIZE = CFG["img_size"]

for split, indices in splits.items():
    img_dir = PROCESSED_DIR / "images" / split
    lbl_dir = PROCESSED_DIR / "labels" / split

    for i, idx in enumerate(tqdm(indices, desc=f"Escrevendo {split}")):
        rec   = all_records[idx]
        frame = cv2.resize(rec["frame"], (IMG_SIZE, IMG_SIZE))
        name  = f"{split}_{i:06d}"

        cv2.imwrite(str(img_dir / f"{name}.jpg"), frame)
        with open(lbl_dir / f"{name}.txt", "w") as f:
            f.write("\n".join(rec["annotations"]))

print("\n✓ Dataset escrito em disco")

In [ ]:
# ── Gerar dataset.yaml ────────────────────────────────────────────────────────
dataset_yaml = {
    "path":  str(PROCESSED_DIR.resolve()),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    3,
    "names": ["person", "violence", "pre_violence"],
}

yaml_path = PROCESSED_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"✓ dataset.yaml salvo em: {yaml_path}")
print()
print(open(yaml_path).read())

In [ ]:
# ── Salvar metadados e features temporais ─────────────────────────────────────
stats = {
    "total_frames": len(all_records),
    "split": {k: len(v) for k, v in splits.items()},
    "class_distribution": {
        "person":       sum(1 for r in all_records if r["class_id"] == 0),
        "violence":     sum(1 for r in all_records if r["class_id"] == 1),
        "pre_violence": sum(1 for r in all_records if r["class_id"] == 2),
    },
    "aggression_score": {
        "mean": float(np.mean([r["aggression_score"] for r in all_records])),
        "std":  float(np.std ([r["aggression_score"] for r in all_records])),
        "max":  float(np.max ([r["aggression_score"] for r in all_records])),
    },
}

with open(META_DIR / "stats.json", "w") as f:
    json.dump(stats, f, indent=2)

# Features temporais para o notebook 03 (LSTM)
# Agrupa frames por vídeo e salva sequência de features
from collections import defaultdict
video_sequences = defaultdict(list)
for r in all_records:
    n_persons = max(1, len(r["annotations"]))
    feat = [
        float(r["class_id"] == 1),          # violence flag
        float(r["class_id"] == 2),          # pre_violence flag
        r["aggression_score"],
        min(1.0, n_persons / 5.0),
        0.5,                                 # placeholder optical flow
    ]
    video_sequences[r["video"]].append({"feat": feat, "label": r["class_id"]})

temporal_data = [
    {"features": [f["feat"] for f in seq], "label": max(f["label"] for f in seq)}
    for seq in video_sequences.values()
]

with open(META_DIR / "temporal_features.json", "w") as f:
    json.dump(temporal_data, f)

print(f"✓ Metadados: {META_DIR / 'stats.json'}")
print(f"✓ Features temporais: {META_DIR / 'temporal_features.json'}")
print(f"  Sequências de vídeo: {len(temporal_data)}")

## 9. Visualização do dataset

In [ ]:
# ── Distribuição de classes ────────────────────────────────────────────────────
dist = stats["class_distribution"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Barra
colors = ["steelblue", "tomato", "darkorange"]
axes[0].bar(dist.keys(), dist.values(), color=colors, edgecolor="black", alpha=0.85)
axes[0].set_title("Distribuição de Classes", fontsize=13)
axes[0].set_ylabel("Número de frames")
for i, (k, v) in enumerate(dist.items()):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Histograma de aggression score
scores = [r["aggression_score"] for r in all_records]
axes[1].hist(scores, bins=30, color="slateblue", edgecolor="black", alpha=0.8)
axes[1].axvline(CFG["pre_violence_threshold"], color="orange", lw=2,
                linestyle="--", label=f"Threshold ({CFG['pre_violence_threshold']})")
axes[1].set_title("Distribuição do Aggression Score", fontsize=13)
axes[1].set_xlabel("Score")
axes[1].set_ylabel("Frequência")
axes[1].legend()

plt.suptitle(f"Dataset — {stats['total_frames']} frames totais", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Amostras do dataset com bboxes ────────────────────────────────────────────
sample_images = list((PROCESSED_DIR / "images" / "train").glob("*.jpg"))[:9]

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for ax, img_path in zip(axes.flat, sample_images):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    lbl_path = PROCESSED_DIR / "labels" / "train" / (img_path.stem + ".txt")

    ax.imshow(img)
    h, w = img.shape[:2]

    if lbl_path.exists():
        for line in lbl_path.read_text().strip().split("\n"):
            if not line:
                continue
            cls, cx, cy, bw, bh = map(float, line.split())
            cls = int(cls)
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            rect = patches.Rectangle(
                (x1, y1), bw * w, bh * h,
                linewidth=2, edgecolor=CLASS_COLORS[cls], facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, CLASS_NAMES[cls], color=CLASS_COLORS[cls],
                    fontsize=9, fontweight="bold")

    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")

plt.suptitle("Amostras do conjunto de treino com anotações", fontsize=13)
plt.tight_layout()
plt.show()

## Resumo

| Item | Valor |
|---|---|
| Total de frames | veja `stats.json` |
| Classes | person / violence / pre_violence |
| Formato | YOLO (txt normalizado) |
| Próximo passo | `02_training_yolov8.ipynb` |